# **Scraping Planos Masterlink**

_Página do provedor: https://masterlinktelecom.net/_

In [ ]:
# Bibliotecas
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [ ]:
# Obter lista de planos
texto_html = open("../../data/scraped/masterlink.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
conteiner_planos = soup.find("div", id="planosnet")
planos = conteiner_planos.find_all("div", class_="e-con-full")
len(planos)

In [ ]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find_all("h2")[0].text
    pre_preco = plano.find_all("h2")
    preco = ".".join([i.text for i in pre_preco])
    pre_sva = plano.find_all("span", class_="elementor-icon-list-text")
    svas = ", ".join([i.text for i in pre_sva])
    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

dados = []

for i in planos:
    dados_plano = raspar_plano(i)
    dados.append(dados_plano)

df = pd.DataFrame(dados)
df = df[df["svas"] != ""]
df

## **2. Tratamento**

In [ ]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Masterlink"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"].str.extract(r"(\d+\.?\d*)")

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"].str.split("$").str[1]
    .str.replace(r"\D", "", regex=True)
    .astype(float)
) / 100

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "MASTERLINK.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")